# The Anatomy of a Machine Learning Problem
Companion to the notes: probabilities from data, the losses, softmax by hand, and overfitting.

## 1. Classification predicts probabilities: the age table

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ages = np.repeat([20, 25, 30, 35], 5)
bought = np.array([1,0,0,0,0, 1,1,0,0,0, 1,1,1,0,0, 1,1,1,1,0])
df = pd.DataFrame({"age": ages, "bought": bought})
print(df.groupby("age")["bought"].agg(["count", "sum", "mean"]))

## 2. Why cross-entropy and not squared error: gradients with respect to z when y = 1

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))
for z in [-8, -3, 0, 3]:
    p = sigmoid(z)
    grad_mse = (p - 1) * p * (1 - p)        # d/dz of 0.5 (y - p)^2 with y = 1
    grad_ce = p - 1                           # d/dz of cross-entropy with y = 1
    print(f"z={z:>3}: p={p:.6f}  MSE gradient={grad_mse:+.6f}  cross-entropy gradient={grad_ce:+.6f}")

## 3. Softmax and categorical cross-entropy by hand

In [ ]:
z = np.array([2.0, 1.0, -0.5])
y = np.array([1, 0, 0])
p = np.exp(z) / np.exp(z).sum()
print("softmax:", p.round(4), " sum:", p.sum())
print("loss:", round(-np.sum(y * np.log(p)), 4))
print("gradient p - y:", (p - y).round(4))

# numerical check of the gradient
def loss(z): 
    q = np.exp(z) / np.exp(z).sum(); return -np.sum(y * np.log(q))
h = 1e-6
num = [(loss(z + h * np.eye(3)[k]) - loss(z - h * np.eye(3)[k])) / (2 * h) for k in range(3)]
print("numerical gradient:", np.round(num, 4))

## 4. Overfitting: polynomials of increasing degree

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(7)
x_train = np.sort(rng.uniform(0, 1, 20))
y_train = np.sin(2 * np.pi * x_train) + rng.normal(0, 0.3, 20)
x_test = rng.uniform(x_train.min(), x_train.max(), 500)
y_test = np.sin(2 * np.pi * x_test) + rng.normal(0, 0.3, 500)

def fit(degree):
    return make_pipeline(PolynomialFeatures(degree), LinearRegression()).fit(x_train[:, None], y_train)

def mse(model, x, y):
    return np.mean((model.predict(x[:, None]) - y) ** 2)

for d in [1, 3, 9, 15]:
    m = fit(d)
    print(f"degree {d:>2}: train MSE {mse(m, x_train, y_train):.3f}   test MSE {mse(m, x_test, y_test):.3f}")

In [ ]:
degrees = range(1, 16)
train_err = [mse(fit(d), x_train, y_train) for d in degrees]
test_err = [mse(fit(d), x_test, y_test) for d in degrees]
plt.plot(degrees, train_err, "o-", label="training error")
plt.plot(degrees, test_err, "s-", label="test error")
plt.yscale("log"); plt.xlabel("polynomial degree"); plt.ylabel("MSE"); plt.legend()

## 5. A train / validation / test split

In [ ]:
from sklearn.model_selection import train_test_split

x_all = rng.uniform(0, 1, 300)
y_all = np.sin(2 * np.pi * x_all) + rng.normal(0, 0.3, 300)
x_trval, x_te, y_trval, y_te = train_test_split(x_all, y_all, test_size=0.2, random_state=1)
x_tr, x_val, y_tr, y_val = train_test_split(x_trval, y_trval, test_size=0.25, random_state=1)

val_scores = {}
for d in range(1, 12):
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(x_tr[:, None], y_tr)
    val_scores[d] = np.mean((m.predict(x_val[:, None]) - y_val) ** 2)
best = min(val_scores, key=val_scores.get)
final = make_pipeline(PolynomialFeatures(best), LinearRegression()).fit(x_trval[:, None], y_trval)
print("best degree on validation:", best)
print("test MSE of the final model:", round(np.mean((final.predict(x_te[:, None]) - y_te) ** 2), 4))

## Try it yourself
1. Gradients for z = -3, 0, 3: where do the two losses differ most?
2. Softmax of [1,1,1] vs [10,10,10]; [1,2,3] vs [2,3,4].
3. Change 20 training points to 200 in section 4. What happens to degree 9?